# Session 4: Reflexion (Learning Agents)
**Goal**: Agent critiques its own work and saves lessons to Episodic Memory.

In [ ]:
import os
import sys
import json
sys.path.insert(0, '..')
import utils.bootstrap

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent
from typing import TypedDict
from utils.tools import draft_email_reply
from utils.llm_router import get_routed_llm

### Setup State & Memory

In [ ]:
class ReflexionState(TypedDict):
    task: str
    current_draft: str
    critique: str
    is_passing: bool
    iterations: int

MEMORY_FILE = 'episodic_memory.json'

def load_memory():
    return json.load(open(MEMORY_FILE)) if os.path.exists(MEMORY_FILE) else []

def save_memory(task, draft, critique):
    mem = load_memory()
    mem.append({"task": task, "draft": draft, "lesson": critique})
    json.dump(mem, open(MEMORY_FILE, 'w'))

### Setup Agents

In [ ]:
llm = get_routed_llm(role='master_model')
actor_agent = create_react_agent(llm, tools=[draft_email_reply], prompt=SystemMessage(content="DRAFTER: Output a professional email draft. Address any critiques."))
evaluator_agent = create_react_agent(llm, tools=[], prompt=SystemMessage(content="EVALUATOR: Grade the draft. It must be < 100 words and polite. Respond ONLY with JSON: {'pass': bool, 'critique': '...'}"))

### Define Graph Nodes

In [ ]:
def actor_node(state: ReflexionState):
    print(f"\n\u270d\ufe0f [ACTOR] Drafting (Iter {state['iterations'] + 1}) ...")
    mem_ctx = "\n".join([m['draft'] for m in load_memory()[-2:]])
    prompt = f"Task: {state['task']}\nPast Good Examples: {mem_ctx}\nCritique: {state['critique']}"
    res = actor_agent.invoke({"messages": [HumanMessage(content=prompt)]})
    return {"current_draft": str(res['messages'][-1].content), "iterations": state['iterations'] + 1}

def evaluator_node(state: ReflexionState):
    print("\n\ud83e\uddd0 [EVALUATOR] Grading...")
    res = evaluator_agent.invoke({"messages": [HumanMessage(content=f"Draft: {state['current_draft']}")]})
    try:
        text = res['messages'][-1].content
        if '```json' in text: text = text.split('```json')[1].split('```')[0]
        elif '```' in text: text = text.split('```')[1].split('```')[0]
        ev = json.loads(text.strip())
        return {"is_passing": ev.get('pass', False), "critique": ev.get('critique', 'Failed.')}
    except: return {"is_passing": False, "critique": "JSON Parse Error"}

def route(state: ReflexionState):
    if state['is_passing']: return "save_node"
    if state['iterations'] >= 3: return END
    return "actor_node"

def save_node(state: ReflexionState):
    print("\n\ud83d\udcbe [MEMORY] Saving to disk...")
    save_memory(state['task'], state['current_draft'], state['critique'])
    return state

### Build & Run Graph

In [ ]:
builder = StateGraph(ReflexionState)
builder.add_node("actor_node", actor_node)
builder.add_node("evaluator_node", evaluator_node)
builder.add_node("save_node", save_node)
builder.add_edge(START, "actor_node")
builder.add_edge("actor_node", "evaluator_node")
builder.add_conditional_edges("evaluator_node", route)
builder.add_edge("save_node", END)
app = builder.compile()

task = "Apologize to a VIP client for a 2-week delay. Extremely polite, under 100 words."
res = app.invoke({"task": task, "current_draft": "", "critique": "", "is_passing": False, "iterations": 0})
print("\n\u2728 Final Draft:\n", res['current_draft'])